In [1]:
# Tune Random Forest on training and validation data; compare with Elastic Net on identical test rows.
# Select settings on validation only and report calibration and test metrics.

In [2]:
import sys as _sys
from pathlib import Path as _Path
_UTILS_ROOT = next(path for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
                   if (path / "notebook_utils.py").is_file())
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths, file_hash, probability_metrics, calibration_table, embedded_image, load_split
from pathlib import Path
import base64
import hashlib
import html
import io
import json
import platform
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.calibration import calibration_curve
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, auc, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, precision_recall_curve,
    roc_auc_score, roc_curve)
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'data/processed/train_matrix.csv').exists(): ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
REPORTS = ROOT / 'artifacts/20_model_random_forest'
MODELS = project_paths(ROOT).models
FIGURES = REPORTS
N_ESTIMATORS, SEED = 400, 42
MAX_DEPTHS = [4, 8, None]
MIN_LEAF_SIZES = [2, 5, 10]
MAX_FEATURES = ['sqrt', .25]
CLASS_WEIGHTS = {'unweighted': None, 'balanced_subsample': 'balanced_subsample'}
# Sequential fits avoid Windows worker-pool IPC requirements.
N_JOBS = 1
input_hashes = {}


artifact_path = MODELS / 'preprocessing_pipeline.joblib'
input_hashes[artifact_path] = file_hash(artifact_path)
preprocessing = joblib.load(artifact_path)
schema = preprocessing.project_schema_
FEATURES, ID_COLUMNS = schema['matrix_feature_columns'], schema['matrix_metadata_columns']
preprocessing_state = joblib.hash(preprocessing)


train, validation = load_split('train', PROCESSED, schema, ID_COLUMNS, FEATURES, input_hashes), load_split('validation', PROCESSED, schema, ID_COLUMNS, FEATURES, input_hashes)
train_before, validation_before = train.copy(deep=True), validation.copy(deep=True)
selector = VarianceThreshold(threshold=0.).fit(train[FEATURES].to_numpy(dtype=float))
keep = selector.get_support()
ACTIVE_FEATURES = np.asarray(FEATURES)[keep].tolist()
EXCLUDED_CONSTANTS = np.asarray(FEATURES)[~keep].tolist()
X_train = selector.transform(train[FEATURES].to_numpy(dtype=float))
X_validation = selector.transform(validation[FEATURES].to_numpy(dtype=float))
y_train, y_validation = train.target.to_numpy(dtype=int), validation.target.to_numpy(dtype=int)
selector_state = joblib.hash(selector)


In [3]:


# Every candidate sees the same training predictors; validation selects the forest.
candidates, tuning_rows = {}, []
for weight_name, class_weight in CLASS_WEIGHTS.items():
    for max_depth in MAX_DEPTHS:
        for min_leaf in MIN_LEAF_SIZES:
            for max_features in MAX_FEATURES:
                candidate_id = f'{weight_name}_depth_{max_depth}_leaf_{min_leaf}_features_{max_features}'
                forest = RandomForestClassifier(n_estimators=N_ESTIMATORS, max_depth=max_depth,
                    min_samples_leaf=min_leaf, max_features=max_features, class_weight=class_weight,
                    bootstrap=True, oob_score=False, random_state=SEED, n_jobs=N_JOBS)
                started = time.perf_counter()
                with warnings.catch_warnings(record=True) as captured:
                    warnings.simplefilter('always')
                    forest.fit(X_train, y_train)
                validation_probability = forest.predict_proba(X_validation)[:, 1]
                eligible = bool(not captured and np.isfinite(validation_probability).all())
                row = {'candidate_id': candidate_id, 'n_estimators': N_ESTIMATORS,
                    'max_depth': 'unlimited' if max_depth is None else str(max_depth),
                    'min_samples_leaf': min_leaf, 'max_features': str(max_features),
                    'class_weight': weight_name, 'eligible': eligible,
                    'total_tree_nodes': sum(tree.tree_.node_count for tree in forest.estimators_),
                    'mean_tree_depth': float(np.mean([tree.tree_.max_depth for tree in forest.estimators_])),
                    'elapsed_seconds': time.perf_counter() - started,
                    'warnings': '; '.join(sorted({str(w.message) for w in captured}))}
                row.update({'validation_' + k: v for k, v in probability_metrics(y_validation, validation_probability).items()})
                tuning_rows.append(row)
                if eligible: candidates[candidate_id] = forest
tuning = pd.DataFrame(tuning_rows)
eligible = tuning.loc[tuning.eligible].sort_values(
    ['validation_pr_auc_average_precision', 'validation_roc_auc', 'validation_brier_score',
     'total_tree_nodes', 'candidate_id'], ascending=[False, False, True, True, True], kind='stable')
if eligible.empty: raise RuntimeError('No eligible forest; inspect fit warnings before test evaluation')
best_row = eligible.iloc[0]
best_model = candidates[best_row.candidate_id]
validation_probability = best_model.predict_proba(X_validation)[:, 1]
tuning['selected'] = tuning.candidate_id.eq(best_row.candidate_id)


In [4]:
# Choose and lock a validation-only threshold using the same rule as Elastic Net.
threshold_candidates = np.unique(np.r_[0., .5, validation_probability, np.nextafter(validation_probability.max(), np.inf)])
threshold_rows = pd.DataFrame([probability_metrics(y_validation, validation_probability, value) for value in threshold_candidates])
threshold_rows['distance_from_half'] = (threshold_rows.threshold - .5).abs()
threshold_rows = threshold_rows.sort_values(
    ['balanced_accuracy', 'sensitivity', 'specificity', 'distance_from_half', 'threshold'],
    ascending=[False, False, False, True, False], kind='stable')
OPERATING_THRESHOLD = float(threshold_rows.iloc[0].threshold)
model_configuration = {'selected_candidate': best_row.candidate_id, 'fit_split': 'train',
    'selection_split': 'validation', 'selection_metric': 'average_precision',
    'threshold_metric': 'balanced_accuracy', 'operating_threshold': OPERATING_THRESHOLD,
    'n_estimators': N_ESTIMATORS, 'max_depth': best_model.max_depth,
    'min_samples_leaf': best_model.min_samples_leaf, 'max_features': best_model.max_features,
    'class_weight': best_model.class_weight, 'seed': SEED, 'bootstrap': True,
    'active_predictors': len(ACTIVE_FEATURES), 'training_constant_predictors': EXCLUDED_CONSTANTS,
    'preprocessing_sha256': input_hashes[artifact_path],
    'training_event_ids': train.metadata_event_id.tolist(),
    'validation_event_ids': validation.metadata_event_id.tolist(),
    'calibration': 'No post-hoc calibrator; sampled-cohort probabilities'}
configuration_state, model_state = joblib.hash(model_configuration), joblib.hash(best_model)

# The test is loaded only after selection. Reuse notebook 19's exact test membership.
test = load_split('test', PROCESSED, schema, ID_COLUMNS, FEATURES, input_hashes)
test_before = test.copy(deep=True)
X_test = selector.transform(test[FEATURES].to_numpy(dtype=float))
frames = {'train': train, 'validation': validation, 'test': test}
probabilities = {'train': best_model.predict_proba(X_train)[:, 1],
    'validation': validation_probability, 'test': best_model.predict_proba(X_test)[:, 1]}
metric_rows, prediction_frames = [], []
for split, frame in frames.items():
    for rule, threshold in [('default_0.5', .5), ('validation_selected', OPERATING_THRESHOLD)]:
        metric_rows.append({'split': split, 'rule': rule,
            **probability_metrics(frame.target.to_numpy(), probabilities[split], threshold)})
    out = frame[ID_COLUMNS].copy()
    out['bankruptcy_probability'] = probabilities[split]
    out['predicted_target_default_0_5'] = (probabilities[split] >= .5).astype(int)
    out['predicted_target_operating'] = (probabilities[split] >= OPERATING_THRESHOLD).astype(int)
    prediction_frames.append(out)
metrics, predictions = pd.DataFrame(metric_rows), pd.concat(prediction_frames, ignore_index=True)


In [5]:
# Read the existing Elastic Net benchmark only after RF decisions are frozen.
elastic_path = MODELS / 'elastic_net_logistic.joblib'
elastic_predictions_path = ROOT / 'artifacts/19_model_elastic_net_logistic/elastic_net_predictions.csv'
elastic_metrics_path = ROOT / 'artifacts/19_model_elastic_net_logistic/elastic_net_metrics.csv'
for path in [elastic_path, elastic_predictions_path, elastic_metrics_path]: input_hashes[path] = file_hash(path)
elastic = joblib.load(elastic_path)
elastic_state = joblib.hash(elastic)
elastic_meta = elastic.project_model_
# Notebook 19 stores Windows-relative paths; normalize separators for lookup.
elastic_input_hashes = {Path(name).as_posix(): digest for name, digest in elastic_meta['input_sha256'].items()}

elastic_predictions = pd.read_csv(elastic_predictions_path, dtype={'company_id': 'string', 'metadata_event_id': 'string'})
elastic_metrics = pd.read_csv(elastic_metrics_path)
elastic_probabilities = {}
for split, frame in frames.items():
    prior = elastic_predictions.loc[elastic_predictions.split.eq(split)]
    elastic_probabilities[split] = prior.bankruptcy_probability.to_numpy(dtype=float)
    for rule, threshold in [('default_0.5', .5), ('validation_selected', elastic_meta['operating_threshold'])]:
        recomputed = probability_metrics(frame.target.to_numpy(), elastic_probabilities[split], threshold)
        stored = elastic_metrics.loc[elastic_metrics.split.eq(split) & elastic_metrics.rule.eq(rule)]
        expected_column = 'predicted_target_default_0_5' if rule == 'default_0.5' else 'predicted_target_operating'

# Comparisons exclude training: each model has the same validation and test observations.
comparison = pd.concat([
    elastic_metrics.loc[elastic_metrics.split.isin(['validation', 'test'])].assign(model='Elastic Net'),
    metrics.loc[metrics.split.isin(['validation', 'test'])].assign(model='Random Forest')], ignore_index=True)
comparison = comparison[['model'] + elastic_metrics.columns.tolist()].sort_values(['split', 'rule', 'model'], kind='stable').reset_index(drop=True)
SCORES = ['roc_auc', 'pr_auc_average_precision', 'balanced_accuracy', 'sensitivity', 'specificity', 'f1', 'brier_score']
delta_rows = []
for (split, rule), block in comparison.groupby(['split', 'rule'], sort=True):
    a, b = block.set_index('model').loc['Random Forest'], block.set_index('model').loc['Elastic Net']
    delta_rows.append({'split': split, 'rule': rule, **{score: a[score] - b[score] for score in SCORES}})
differences = pd.DataFrame(delta_rows)


In [6]:


# Training impurity importance is descriptive, not evidence of held-out superiority.
importance = np.zeros(len(FEATURES))
importance[keep] = best_model.feature_importances_
importance_rows = []
for feature, value, active in zip(FEATURES, importance, keep):
    branch, source = feature.split('__', 1)
    family = ('missing_' + source.removeprefix('missingindicator_').split('_', 1)[0] if branch == 'missing'
              else 'categorical_' + ('sector' if source.startswith('sector_') else 'industry') if branch == 'categorical'
              else source.split('_', 1)[0])
    importance_rows.append({'feature': feature, 'family': family, 'included_in_model': bool(active),
        'excluded_reason': '' if active else 'constant in training', 'training_impurity_importance': float(value)})
importances = pd.DataFrame(importance_rows).sort_values(['training_impurity_importance', 'feature'], ascending=[False, True], kind='stable')
family_importance = importances.groupby('family', sort=True).agg(candidate_features=('feature', 'size'),
    active_features=('included_in_model', 'sum'), training_impurity_importance=('training_impurity_importance', 'sum')).reset_index()
final_pipeline = Pipeline([('preprocessing', preprocessing), ('training_constant_selector', selector), ('classifier', best_model)])
final_pipeline.project_model_ = dict(model_configuration)
final_pipeline.project_model_.update({'matrix_feature_columns': FEATURES, 'active_feature_columns': ACTIVE_FEATURES,
    'elastic_net_artifact_sha256': input_hashes[elastic_path],
    'versions': {'python': platform.python_version(), 'sklearn': sklearn.__version__,
                 'numpy': np.__version__, 'pandas': pd.__version__, 'joblib': joblib.__version__}})

# Show held-out curves side by side; the test cannot change any setting.
for directory in [REPORTS, MODELS]: directory.mkdir(parents=True, exist_ok=True)
colors = {'Elastic Net': '#1769aa', 'Random Forest': '#d46c00'}
fig, axes = plt.subplots(2, 2, figsize=(12.5, 10), constrained_layout=True)
for row, split in enumerate(['validation', 'test']):
    labels = frames[split].target.to_numpy()
    for name, probs in [('Elastic Net', elastic_probabilities[split]), ('Random Forest', probabilities[split])]:
        fpr, tpr, _ = roc_curve(labels, probs)
        precision, recall, _ = precision_recall_curve(labels, probs)
        axes[row, 0].plot(fpr, tpr, color=colors[name], label=f'{name}: AUC={roc_auc_score(labels, probs):.3f}')
        axes[row, 1].plot(recall, precision, color=colors[name], label=f'{name}: AP={average_precision_score(labels, probs):.3f}')
    axes[row, 0].plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1)
    axes[row, 1].axhline(labels.mean(), linestyle=':', color='black', label=f'prevalence={labels.mean():.3f}')
    axes[row, 0].set(xlabel='False positive rate', ylabel='Sensitivity', title=f'{split.title()} ROC (n={len(labels)})')
    axes[row, 1].set(xlabel='Recall / sensitivity', ylabel='Precision', title=f'{split.title()} precision-recall')
    for ax in axes[row]: ax.set(xlim=(0, 1), ylim=(0, 1.02)); ax.legend(fontsize=9); ax.grid(alpha=.2)
fig.suptitle('Elastic Net versus Random Forest: identical chronological cohorts')
roc_pr_path = FIGURES / 'random_forest_vs_elastic_net_roc_pr.png'
fig.savefig(roc_pr_path, dpi=180, bbox_inches='tight'); plt.close(fig)

calibration_frames = []
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.3), constrained_layout=True)
for ax, split in zip(axes, ['validation', 'test']):
    labels = frames[split].target.to_numpy()
    for name, probs in [('Elastic Net', elastic_probabilities[split]), ('Random Forest', probabilities[split])]:
        table = calibration_table(labels, probs)
        table['split'], table['model'] = split, name
        calibration_frames.append(table)
        ax.plot(table.mean_predicted_probability, table.observed_bankruptcy_rate,
            marker='o', color=colors[name], label=f'{name} (Brier={brier_score_loss(labels, probs):.3f})')
    ax.plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1, label='perfect calibration')
    ax.set(xlabel='Mean predicted bankruptcy probability', ylabel='Observed bankruptcy fraction',
           title=f'{split.title()} reliability: up to eight quantile bins', xlim=(-.02, 1.02), ylim=(-.02, 1.02))
    ax.legend(fontsize=9); ax.grid(alpha=.2)
fig.suptitle('Sampled-cohort calibration: validation and held-out test')
calibration_path = FIGURES / 'random_forest_vs_elastic_net_calibration.png'
fig.savefig(calibration_path, dpi=180, bbox_inches='tight'); plt.close(fig)
calibration_bins = pd.concat(calibration_frames, ignore_index=True)


In [7]:
# Generate the report using held-out performance for every model comparison.
def table_html(frame):
    return '<div class="table-scroll">' + frame.to_html(index=False, escape=True,
        float_format=lambda value: f'{value:.5g}', na_rep='not available', border=0) + '</div>'


split_summary = pd.DataFrame([{'split': split, 'events': len(frame), 'companies': frame.company_id.nunique(),
    'bankrupt_events': int(frame.target.sum()), 'prevalence': frame.target.mean(),
    'first_prediction': frame.prediction_date.min(), 'last_prediction': frame.prediction_date.max()}
    for split, frame in frames.items()])
configuration_table = pd.DataFrame([{'parameter': key, 'value': str(model_configuration[key])} for key in
    ['selected_candidate', 'n_estimators', 'max_depth', 'min_samples_leaf', 'max_features', 'class_weight', 'operating_threshold', 'active_predictors']])
provenance = pd.DataFrame([{'input': str(path.relative_to(ROOT)), 'sha256': digest} for path, digest in input_hashes.items()])
report_sections = [
    '<h1>Random Forest nonlinear benchmark</h1>',
    '<p>Random Forest is compared with the existing Elastic Net benchmark on identical chronological validation/test '
    'events and labels, with identical training-fitted preprocessing and predictor eligibility. All forest fits use '
    'training only; hyperparameters and threshold are chosen on validation and frozen before the test is opened. '
    'Elastic Net remains the primary interpretable model. No superiority conclusion uses training performance.</p>',
    '<h2>Chronological cohort parity</h2>', table_html(split_summary),
    '<p>Company groups and company IDs are disjoint across splits; prior outcome windows end before the next cohort. '
    'Membership, row order, labels, matrix hashes, active features and frozen preprocessing are checked against '
    'notebook 19. Neither benchmark is retrained for this comparison. Training-constant columns are excluded; '
    'correlated predictors are retained. No clipping or winsorization is applied.</p>',
    '<h2>Random Forest selection protocol</h2>', table_html(configuration_table),
    f'<p>Grid: depths={MAX_DEPTHS}; minimum leaves={MIN_LEAF_SIZES}; feature fractions={MAX_FEATURES}; '
    f'weights={list(CLASS_WEIGHTS)}; trees fixed at {N_ESTIMATORS}; seed={SEED}. '
    f'{int(tuning.eligible.sum())} of {len(tuning)} fits were eligible. Selection ranks validation average precision, '
    'ROC-AUC, lower Brier, fewer total nodes, and candidate ID. No test metric enters ranking. The operating threshold '
    'maximizes validation balanced accuracy, with the same sensitivity/specificity/proximity-to-0.5 tie-breakers as '
    'Elastic Net. The final forest remains fitted on training only.</p>',
    '<h2>Validation and test comparison</h2>', table_html(comparison),
    '<p>Each model uses its own validation-selected threshold for the primary operating comparison. Both are also '
    'reported at a common 0.5 threshold. ROC-AUC, PR-AUC and Brier are threshold independent. PR-AUC primary summary '
    'is average precision; trapezoidal PR area is separately reported. Sensitivity is bankruptcy recall; specificity '
    'is non-bankruptcy recall. Brier is unweighted mean squared probability error.</p>',
    '<h2>Paired cohort metric differences (Random Forest minus Elastic Net)</h2>', table_html(differences),
    '<p>Positive differences favor Random Forest for discrimination/classification scores; negative Brier differences '
    'favor Random Forest for probability accuracy. These are descriptive point estimates. Validation is reused for '
    'selection and test reflects one historical cohort, so these results do not establish statistical significance '
    'or general superiority. Do not choose or retune a model using these test results.</p>',
    embedded_image(roc_pr_path, 'ROC and PR comparisons on the same validation and test rows.'),
    '<h2>Calibration</h2>', embedded_image(calibration_path, 'Empirical reliability curves; no held-out calibration fit.'),
    table_html(calibration_bins),
    '<p>Quantile ties may create fewer than eight bins; bin counts are shown in the table. Brier and reliability '
    'describe this sampled matched cohort. Class weights and case-control sampling mean probabilities are not '
    'population bankruptcy incidence. No post-hoc calibrator uses validation/test outcomes.</p>',
    '<details><summary>Training-only fit diagnostics (not a comparison or selection criterion)</summary>'
        + table_html(metrics.loc[metrics.split.eq('train')]) +
        '<p>Training scores are in-sample and can be optimistic. They are not evidence that Random Forest is better.</p></details>',
    '<h2>Training impurity importances</h2>', table_html(family_importance), table_html(importances.head(30)),
    '<p>Impurity importance is a training diagnostic, biased toward predictors offering many splits. Correlated '
    'variables may divide or substitute for each other; neither individual rank nor family sums imply causality '
    'or held-out usefulness. Full values and excluded constants are in random_forest_feature_importances.csv.</p>',
    '<h2>Validation tuning audit</h2>', table_html(tuning.sort_values(['eligible', 'validation_pr_auc_average_precision'], ascending=[False, False])),
    '<details><summary>Top ten validation operating thresholds</summary>' + table_html(threshold_rows.head(10)) + '</details>',
    '<h2>Artifacts and scope</h2>',
    '<p>models/random_forest.joblib is a raw-input sklearn Pipeline with frozen preprocessing, training-constant '
    'selector, and forest. project_model_ stores the operating threshold, feature names, event IDs, configuration '
    'and provenance. Apply the stored threshold to predict_proba for the selected operating rule; predict follows '
    'the classifier default (class 0 wins an exact probability tie). Inputs and the Elastic Net artifact remain unchanged.</p>',
    '<p>Earlier evidence gaps remain: the requested 1:2 design differs from the existing 1:3 source, time grouping '
    'can break matched sets, independent negative-outcome surveillance is incomplete, and historical entity '
    'continuity/classifications and macro release vintages are not fully verified. This benchmark does not resolve them.</p>',
    table_html(provenance), '<p>Software: ' + html.escape(json.dumps(final_pipeline.project_model_['versions'], sort_keys=True)) + '</p>'
]
report_html = ('<!doctype html><html lang="en"><head><meta charset="utf-8">'
    '<meta name="viewport" content="width=device-width,initial-scale=1"><title>Random Forest benchmark comparison</title>'
    '<style>body{font:15px/1.55 system-ui,sans-serif;margin:32px auto;padding:0 24px;max-width:1280px;color:#172b3a}'
    'h1,h2{color:#123b59}p{max-width:1100px}.table-scroll{overflow:auto;max-height:560px;margin:16px 0}'
    'table{border-collapse:collapse;font-size:12px;width:100%}th,td{padding:7px;text-align:left;border-bottom:1px solid #d8e1e8;white-space:nowrap}'
    'th{position:sticky;top:0;background:#eaf1f6}tr:nth-child(even){background:#f7f9fb}'
    'figure{margin:24px 0}img{max-width:100%;height:auto}figcaption{font-size:13px;color:#52636e}'
    'details{margin:16px 0}summary{cursor:pointer}</style></head><body>' + '\n'.join(report_sections) + '</body></html>')


In [8]:
# Check structural parity, frozen fits and independently reproducible raw-input inference.

raw_path = PROCESSED / 'modeling_dataset_unprocessed.csv'
input_hashes[raw_path] = file_hash(raw_path)
raw = pd.read_csv(raw_path, usecols=['company_id', 'prediction_date'] + schema['input_feature_columns'], dtype={'company_id': 'string'})
raw = raw.set_index(['company_id', 'prediction_date'])
for split, frame in frames.items():
    rows = raw.reindex(pd.MultiIndex.from_frame(frame[['company_id', 'prediction_date']]))
    inputs = rows[schema['input_feature_columns']]
final_pipeline.project_model_.update({'input_sha256': {str(path.relative_to(ROOT)): digest for path, digest in input_hashes.items()},
    'test_event_ids': test.metadata_event_id.tolist()})
model_path = MODELS / 'random_forest.joblib'
joblib.dump(final_pipeline, model_path)
loaded = joblib.load(model_path)
probe = raw.reindex(pd.MultiIndex.from_frame(train.iloc[:12][['company_id', 'prediction_date']]))
for frame, filename in [(tuning, 'random_forest_tuning_results.csv'), (metrics, 'random_forest_metrics.csv'),
    (predictions, 'random_forest_predictions.csv'), (importances, 'random_forest_feature_importances.csv'),
    (comparison, 'random_forest_vs_elastic_net.csv')]: frame.to_csv(REPORTS / filename, index=False)
(REPORTS / 'random_forest_report.html').write_text(report_html, encoding='utf-8')


626803